# Train partwise — v4 22/11/26
Mở notebook trong Google Colab, chạy các ô từ trên xuống. Có thể dùng CPU.
Notebook đọc trực tiếp sheet, tải PKL/GT bằng tài khoản Google của bạn, học trọng số linear cho mô hình partwise hiện có.
Checkpoint từng epoch và bộ tham số tốt nhất lưu trên **Google Drive**.

**Phạm vi:** GT được dùng để chọn track và ghép thời gian khi chuẩn bị dữ liệu có giám sát.
Đây không phải benchmark đồng bộ/nhận dạng độc lập với GT.
Validation là 12 body joints, root-centered MPJPE với một rotation GT cho mỗi đoạn camera;
điểm này không tương đương MPJPE camera có extrinsics.
Không có test độc lập. Kiểm tra alignment.json trước khi tin kết quả.


In [ ]:
# 1. Môi trường và phiên bản code
import os, sys, subprocess, json
from pathlib import Path
REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"
COMMIT = "817dbb0c01ce0354d4a2c98a7fc4b50fc9720bb0"
REPO_DIR = Path("/content/optimization_model_monocular_3")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2.3", "scipy", "torch", "joblib", "pyyaml", "trimesh",
                "gspread", "google-api-python-client", "matplotlib"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


Trainer và module partwise được nhúng ngay trong notebook, nên không cần push các file đang làm dở lên GitHub.

In [ ]:
# 2. Cài snapshot trainer đã kiểm tra
SNAPSHOTS = {'train_v4.py': '"""Supervised calibration of partwise weights from the v4 Sheets export.\n\nGT is used for track/time association and training targets, never for inference.\nAlignment reports must be reviewed before interpreting validation scores.\n"""\nimport argparse\nimport copy\nimport hashlib\nimport itertools\nimport json\nimport random\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nimport yaml\n\nfrom fusion_pipeline.aotr import _ordered_bones\nfrom fusion_pipeline.partwise import (\n    FEATURE_NAMES, blend_frame, eval_weight_model, extract_pair_features,\n    fuse_partwise, slave_directions,\n)\nfrom keypoints_map import load_keypoints3d_map\n\nROOT = Path(__file__).resolve().parent\nBODY = ("left_shoulder", "right_shoulder", "left_elbow", "right_elbow",\n        "left_wrist", "right_wrist", "left_hip", "right_hip",\n        "left_knee", "right_knee", "left_ankle", "right_ankle")\n\n\ndef validate_rows(rows):\n    groups, seen, gt_splits = {}, set(), {}\n    for row in rows:\n        action, cam, split = (str(row.get(k, "")).strip() for k in ("action", "cam", "split"))\n        if not action or not cam.startswith("cam") or not cam[3:].isdigit():\n            raise ValueError(f"Invalid action/camera: {row}")\n        if action.endswith("_" + cam) or split not in {"train", "val"}:\n            raise ValueError(f"Normalize action and split first: {action}/{cam}/{split}")\n        if (action, cam) in seen:\n            raise ValueError(f"Duplicate camera: {action}/{cam}")\n        seen.add((action, cam))\n        if not row.get("file PKL") or not row.get("GT"):\n            raise ValueError(f"Missing PKL/GT: {action}/{cam}")\n        for key, mapping in ((action, groups), (str(row["GT"]), gt_splits)):\n            previous = mapping.setdefault(key, split)\n            if previous != split:\n                raise ValueError(f"Train/val leakage: {key}")\n    if set(groups.values()) != {"train", "val"}:\n        raise ValueError("Both train and val sequences are required")\n    for action in groups:\n        if sum(r["action"] == action for r in rows) < 2:\n            raise ValueError(f"Need two cameras: {action}")\n    return groups\n\n\ndef file_id(value):\n    import re\n    value = str(value).strip()\n    if "/" in value or "?" in value:\n        match = re.search(r"/d/([\\w-]+)|[?&]id=([\\w-]+)", value)\n        if not match:\n            raise ValueError(f"Not a Drive file ID/URL: {value}")\n        return next(x for x in match.groups() if x)\n    if not re.fullmatch(r"[\\w-]{10,}", value):\n        raise ValueError(f"Invalid Drive file ID: {value}")\n    return value\n\n\ndef download(drive, value, cache):\n    from googleapiclient.http import MediaIoBaseDownload\n    cache.mkdir(parents=True, exist_ok=True)\n    path = cache / file_id(value)\n    if not path.exists():\n        temp = path.with_suffix(".download")\n        with temp.open("wb") as stream:\n            reader = MediaIoBaseDownload(stream, drive.files().get_media(fileId=path.name))\n            done = False\n            while not done:\n                _, done = reader.next_chunk()\n        temp.replace(path)\n    return path\n\n\ndef joints_from_vertices(vertices, regressor):\n    vertices = np.asarray(vertices)\n    if vertices.ndim != 3 or vertices.shape[1:] != (6890, 3):\n        raise ValueError(f"Expected [T,6890,3] vertices, got {vertices.shape}")\n    result = np.einsum("jv,tvc->tjc", regressor, vertices, optimize=True)\n    if not np.isfinite(result).all():\n        raise ValueError("Non-finite joint coordinates")\n    return result.astype(np.float32)\n\n\ndef centered(a, names):\n    return a - a[:, [names.index("left_hip"), names.index("right_hip")]].mean(1)[:, None]\n\n\ndef rotation(source, target):\n    u, _, vt = np.linalg.svd(source.reshape(-1, 3).T @ target.reshape(-1, 3))\n    correction = np.eye(3)\n    correction[-1, -1] = np.linalg.det(u @ vt)\n    return u @ correction @ vt\n\n\ndef pa_errors(source, target):\n    source = source - source.mean(1, keepdims=True)\n    target = target - target.mean(1, keepdims=True)\n    u, _, vt = np.linalg.svd(np.einsum("tji,tjk->tik", source, target))\n    correction = np.broadcast_to(np.eye(3), u.shape).copy()\n    correction[:, -1, -1] = np.linalg.det(u @ vt)\n    rot = u @ correction @ vt\n    moved = source @ rot\n    scale = (moved * target).sum((1, 2)) / np.maximum((moved * moved).sum((1, 2)), 1e-12)\n    return np.linalg.norm(moved * scale[:, None, None] - target, axis=-1).mean(1)\n\n\nclass AlignmentError(ValueError):\n    """A camera has insufficient overlap or pose agreement with GT."""\n\n\ndef associate(tracks, gt, names, ratios=(0.5, 1.0, 2.0)):\n    """GT-assisted timeline search and track stitching with a switch penalty."""\n    body = [names.index(n) for n in BODY]\n    keys = sorted(tracks)\n    all_ids = np.unique(np.concatenate([np.asarray(tracks[k][0], dtype=int) for k in keys]))\n    if len(all_ids) < 16:\n        raise AlignmentError("Need >=16 camera frames for alignment")\n    # Precompute PA costs against every GT tick once, then search timelines cheaply.\n    costs = np.full((len(all_ids), len(keys), len(gt)), np.inf)\n    joints_at = {}\n    for ki, key in enumerate(keys):\n        ids, joints = tracks[key]\n        ids = np.asarray(ids, dtype=int)\n        if len(np.unique(ids)) != len(ids) or len(ids) != len(joints):\n            raise ValueError(f"Invalid/duplicate frame ids for track {key}")\n        positions = np.searchsorted(all_ids, ids)\n        source = np.repeat(joints[:, None, body], len(gt), axis=1)\n        target = np.broadcast_to(gt[None, :, body], source.shape)\n        costs[positions, ki] = pa_errors(source.reshape(-1, len(body), 3),\n                                        target.reshape(-1, len(body), 3)).reshape(len(ids), len(gt))\n        for pos, joint in zip(positions, joints):\n            joints_at[(int(pos), ki)] = joint\n    roots = np.zeros((len(all_ids), len(keys), 3))\n    for (pos, ki), joints in joints_at.items():\n        roots[pos, ki] = joints[[names.index("left_hip"), names.index("right_hip")]].mean(0)\n    continuity = {}\n    candidates = []\n    for ratio in ratios:\n        mapped = np.rint((all_ids - all_ids.min()) * ratio).astype(int)\n        for offset in range(-int(mapped.max()), len(gt)):\n            gi = mapped + offset\n            keep = np.where((gi >= 0) & (gi < len(gt)))[0]\n            if len(keep) < max(16, int(np.ceil(0.7 * len(all_ids)))):\n                continue\n            unique_gt, unique = np.unique(gi[keep], return_index=True)\n            positions = keep[unique]\n            coverage = min(1.0, (float(np.ptp(unique_gt)) + ratio) / len(gt))\n            if len(unique_gt) < 16 or coverage < 0.5:\n                continue\n            local = costs[positions, :, unique_gt]\n            previous = local[0].copy()\n            parents = []\n            for step, emission in enumerate(local[1:], 1):\n                link = (int(positions[step - 1]), int(positions[step]))\n                if link not in continuity:\n                    distance = np.linalg.norm(roots[link[0], :, None] - roots[link[1], None, :], axis=-1)\n                    continuity[link] = (1 - np.eye(len(keys))) * (0.025 + 0.1 * np.minimum(distance, 5))\n                options = previous[:, None] + continuity[link]\n                parent = np.argmin(options, axis=0)\n                previous = emission + options[parent, np.arange(len(keys))]\n                parents.append(parent)\n            final = int(np.argmin(previous))\n            if not np.isfinite(previous[final]):\n                continue\n            path = [final]\n            for parent in reversed(parents):\n                path.append(int(parent[path[-1]]))\n            path = np.array(path[::-1])\n            raw_error = float(np.mean(local[np.arange(len(path)), path]))\n            score = float(previous[final] / len(path) + 0.03 * (1 - coverage))\n            candidates.append((score, raw_error, ratio, offset, positions, unique_gt, path, coverage))\n    if not candidates:\n        raise AlignmentError("No alignment covers >=70% camera frames and >=50% GT timeline")\n    candidates.sort(key=lambda c: c[:4])\n    score, error, ratio, offset, positions, gi, path, coverage = candidates[0]\n    if error > 0.15:\n        raise AlignmentError(f"Alignment PA error {error * 1000:.1f}mm exceeds 150mm; inspect data")\n    joints = np.array([joints_at[(int(pos), int(ki))] for pos, ki in zip(positions, path)])\n    segments = []\n    for i, ki in enumerate(path):\n        key = keys[ki]\n        if not segments or segments[-1]["track"] != key:\n            segments.append({"track": key, "gt_start": int(gi[i]), "gt_end": int(gi[i])})\n        else:\n            segments[-1]["gt_end"] = int(gi[i])\n    report = dict(track=segments[0]["track"] if len(segments) == 1 else "stitched",\n                  segments=segments, ratio=ratio, offset=offset, matched=len(gi),\n                  coverage=coverage, pa_mm=error * 1000,\n                  switches=len(segments)-1,\n                  runner_up_score_mm=candidates[1][0] * 1000 if len(candidates) > 1 else None)\n    return gi, joints, report\n\n\ndef synchronize(ids_m, jm, ids_s, js):\n    """Interpolate cameras onto shared GT ticks; never bridge missing track gaps."""\n    common = np.arange(max(ids_m.min(), ids_s.min()), min(ids_m.max(), ids_s.max()) + 1)\n    arrays, valid = [], np.ones(len(common), dtype=bool)\n    for ids, joints in ((ids_m, jm), (ids_s, js)):\n        hi = np.minimum(np.searchsorted(ids, common), len(ids) - 1)\n        lo = np.maximum(hi - 1, 0)\n        step = max(1.0, float(np.median(np.diff(ids))))\n        valid &= (common == ids[hi]) | ((ids[hi] - ids[lo]) <= 1.5 * step)\n        flat = joints.reshape(len(ids), -1)\n        values = np.stack([np.interp(common, ids, column) for column in flat.T], axis=1)\n        arrays.append(values.reshape(len(common), *joints.shape[1:]).astype(np.float32))\n    return common[valid], arrays[0][valid], arrays[1][valid]\n\n\ndef prepare(rows, drive, cache, config, alignment_path=None, skip_unaligned_actions=False):\n    validate_rows(rows)\n    mapping = load_keypoints3d_map(str(ROOT / "configs/keypoints3D_map.yml"))\n    names = [k["name"] for k in mapping["keypoints"]]\n    reg = np.load(ROOT / "models/J_regressor_body25_extended.npy")\n    reg = reg[[k["regressor_index"] for k in mapping["keypoints"]]]\n    pairs, reports = [], []\n    for action in dict.fromkeys(r["action"] for r in rows):\n        cameras = []\n        group = sorted((r for r in rows if r["action"] == action), key=lambda r: int(r["cam"][3:]))\n        if len({file_id(r["GT"]) for r in group}) != 1:\n            raise ValueError(f"Different GT files in one sequence: {action}")\n        with np.load(download(drive, group[0]["GT"], cache), allow_pickle=False) as data:\n            gt = joints_from_vertices(data["verts"], reg)\n        for row in group:\n            raw = joblib.load(download(drive, row["file PKL"], cache))\n            tracks = {}\n            for key, track in raw.items():\n                if isinstance(track, dict) and "verts" in track and "frame_ids" in track:\n                    tracks[str(key)] = (track["frame_ids"], joints_from_vertices(track["verts"], reg))\n            if not tracks:\n                raise ValueError(f"Expected WHAM tracks with camera-frame verts: {action}/{row[\'cam\']}")\n            print(f"Aligning {action}/{row[\'cam\']}", flush=True)\n            try:\n                gi, joints, report = associate(tracks, gt, names)\n            except AlignmentError as exc:\n                frame_ids = np.unique(np.concatenate([track[0] for track in tracks.values()]))\n                report = dict(status="rejected", reason=str(exc), coverage=None, pa_mm=None,\n                              camera_frames=len(frame_ids), gt_frames=len(gt),\n                              frame_start=int(frame_ids.min()) if len(frame_ids) else None,\n                              frame_end=int(frame_ids.max()) if len(frame_ids) else None)\n            else:\n                report = dict(status="accepted", **report)\n                cameras.append((row["cam"], gi, joints))\n            reports.append(dict(action=action, cam=row["cam"], **report))\n            print(json.dumps(reports[-1]), flush=True)\n            if alignment_path is not None:\n                atomic_json(Path(alignment_path), reports)\n        for (cam_m, ids_m, jm), (cam_s, ids_s, js) in itertools.combinations(cameras, 2):\n            common, synced_m, synced_s = synchronize(ids_m, jm, ids_s, js)\n            # Temporal features/stability must never bridge missing GT ticks.\n            stride = float(np.median(np.diff(common))) if len(common) > 1 else 1\n            runs = np.split(np.arange(len(common)), np.where(np.diff(common) != stride)[0] + 1)\n            for run in runs:\n                if len(run) < 8:\n                    continue\n                for cm, cs, master, slave in ((cam_m, cam_s, synced_m[run], synced_s[run]),\n                                              (cam_s, cam_m, synced_s[run], synced_m[run])):\n                    target = centered(gt[common[run]], names)\n                    master_c = centered(master, names)\n                    body = [names.index(n) for n in BODY]\n                    target = target @ rotation(target[:, body], master_c[:, body])\n                    M = [dict(zip(names, f)) for f in master]\n                    S = [dict(zip(names, f)) for f in slave]\n                    feats, children = extract_pair_features(M, S)\n                    directions = slave_directions(M, S, config.get("rotation_half_window", 4))\n                    pairs.append(dict(action=action, split=group[0]["split"], cameras=[cm, cs],\n                                      names=names, M=M, S=S, features=feats, children=children,\n                                      directions=directions, target=target.astype(np.float32)))\n        if not any(p["action"] == action for p in pairs):\n            reason = (f"{action}: need at least two aligned cameras; got {len(cameras)}."\n                      if len(cameras) < 2 else f"No synchronized segments for {action}.")\n            if not skip_unaligned_actions:\n                raise ValueError(reason + " Inspect rejected cameras in alignment.json and correct their data/timeline.")\n            reports.append(dict(action=action, split=group[0]["split"], cam=None,\n                                status="skipped_action", reason=reason, coverage=None, pa_mm=None))\n            print(json.dumps(reports[-1]), flush=True)\n            if alignment_path is not None:\n                atomic_json(Path(alignment_path), reports)\n            continue\n        print(f"Prepared {action}: {len(cameras)} cameras", flush=True)\n    if {p["split"] for p in pairs} != {"train", "val"}:\n        raise ValueError("No usable pairs for one of the splits; inspect alignment/coverage")\n    return pairs, reports\n\n\ndef export_model(weights, std):\n    return dict(type="linear", feature_names=list(FEATURE_NAMES), mean=[0.0] * len(std),\n                std=np.asarray(std).tolist(), weights=np.asarray(weights).tolist())\n\n\ndef torch_fuse(pair, weights, std):\n    names = pair["names"]\n    master = torch.as_tensor(np.array([[f[n] for n in names] for f in pair["M"]]), dtype=torch.float32)\n    features = torch.as_tensor(pair["features"])\n    w = torch.sigmoid((features / std) @ weights)\n    result = {n: master[:, i] for i, n in enumerate(names)}\n    for child, parent in _ordered_bones(pair["M"][0]):\n        bi = pair["children"].index(child)\n        direction = torch.tensor(np.array([d.get(child, np.zeros(3)) for d in pair["directions"]]), dtype=torch.float32)\n        m = master[:, names.index(child)] - master[:, names.index(parent)]\n        length = torch.linalg.vector_norm(m, dim=1, keepdim=True)\n        unit = w[:, bi, None] * m / length.clamp_min(1e-9) + (1 - w[:, bi, None]) * direction\n        norm = torch.linalg.vector_norm(unit, dim=1, keepdim=True)\n        unit = torch.where(norm < 1e-9, m / length.clamp_min(1e-9), unit / norm.clamp_min(1e-9))\n        delta = torch.where(length < 1e-9, m, unit * length)\n        result[child] = result[parent] + delta\n    output = torch.stack([result[n] for n in names], dim=1)\n    root = output[:, [names.index("left_hip"), names.index("right_hip")]].mean(1, keepdim=True)\n    return output - root\n\n\ndef validation(pairs, model, config):\n    # Equal sequence weight prevents sequences with more camera pairs dominating selection.\n    scores, baselines = {}, {}\n    for p in pairs:\n        cfg = copy.deepcopy(config)\n        cfg["weight_model"] = model\n        fused, _ = fuse_partwise(p["M"], p["S"], cfg)\n        names = p["names"]\n        inds = [names.index(n) for n in BODY]\n        output = centered(np.array([[f[n] for n in names] for f in fused]), names)\n        master = centered(np.array([[f[n] for n in names] for f in p["M"]]), names)\n        for values, pred in ((scores, output), (baselines, master)):\n            values.setdefault(p["action"], []).append(float(np.linalg.norm(\n                pred[:, inds] - p["target"][:, inds], axis=-1).mean() * 1000))\n    return (float(np.mean([np.mean(v) for v in scores.values()])),\n            float(np.mean([np.mean(v) for v in baselines.values()])))\n\n\ndef atomic_json(path, value):\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    temp.replace(path)\n\n\ndef fit(pairs, config, out, epochs=50, lr=0.01, resume=None, identity=""):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    train = [p for p in pairs if p["split"] == "train"]\n    val = [p for p in pairs if p["split"] == "val"]\n    if not train or not val:\n        raise ValueError("Both splits need usable pairs")\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    # Antisymmetry requires zero mean; estimate scale on train only.\n    flat = np.concatenate([p["features"].reshape(-1, len(FEATURE_NAMES)) for p in train])\n    std = torch.tensor(np.maximum(np.sqrt(np.mean(flat ** 2, axis=0)), 1e-6))\n    weights = torch.nn.Parameter(torch.zeros(len(FEATURE_NAMES)))\n    optimizer = torch.optim.Adam([weights], lr=lr)\n    signature = hashlib.sha256((identity + json.dumps(config, sort_keys=True) +\n                               str(lr)).encode()).hexdigest()\n    history, best, start = [], float("inf"), 1\n    if resume:\n        ckpt = torch.load(resume, map_location="cpu", weights_only=False)\n        if ckpt["signature"] != signature or not np.allclose(ckpt["std"], std.numpy()):\n            raise ValueError("Resume dataset/config/lr differs")\n        weights.data.copy_(ckpt["weights"])\n        optimizer.load_state_dict(ckpt["optimizer"])\n        history, best, start = ckpt["history"], ckpt["best_val_mm"], ckpt["epoch"] + 1\n        random.setstate(ckpt["random_state"])\n        torch.set_rng_state(ckpt["torch_rng"])\n        # Restore the selected model even when resuming into a new output folder.\n        atomic_json(out / "best_weights.json", ckpt["best_model"])\n    elif list(out.glob("epoch_*.pt")):\n        raise ValueError("Output already has checkpoints; resume or choose a new folder")\n    best_model = ckpt["best_model"] if resume else None\n    actions = sorted({p["action"] for p in train})\n    for epoch in range(start, epochs + 1):\n        order = actions.copy()\n        random.shuffle(order)\n        losses = []\n        for action in order:\n            group = [p for p in train if p["action"] == action]\n            optimizer.zero_grad()\n            action_losses = []\n            for p in group:\n                inds = [p["names"].index(n) for n in BODY]\n                target = torch.as_tensor(p["target"])\n                loss = torch.linalg.vector_norm(torch_fuse(p, weights, std)[:, inds] - target[:, inds], dim=-1).mean()\n                (loss / len(group)).backward()\n                action_losses.append(float(loss.detach()) * 1000)\n            optimizer.step()\n            losses.append(float(np.mean(action_losses)))\n        model = export_model(weights.detach().numpy(), std.numpy())\n        score, baseline = validation(val, model, config)\n        if not np.isfinite(score):\n            raise ValueError("Non-finite validation score")\n        improved = score < best\n        if improved:\n            best, best_model = score, dict(model, epoch=epoch, val_mpjpe_mm=score,\n                                           baseline_val_mpjpe_mm=baseline)\n        history.append(dict(epoch=epoch, train_mpjpe_mm=float(np.mean(losses)),\n                            val_mpjpe_mm=score, baseline_val_mpjpe_mm=baseline))\n        checkpoint = dict(epoch=epoch, weights=weights.detach().clone(), std=std.numpy(),\n                          optimizer=optimizer.state_dict(), signature=signature, history=history,\n                          best_val_mm=best, best_model=best_model,\n                          random_state=random.getstate(), torch_rng=torch.get_rng_state())\n        temp = out / "checkpoint.tmp"\n        torch.save(checkpoint, temp)\n        temp.replace(out / f"epoch_{epoch:04d}.pt")\n        atomic_json(out / f"epoch_{epoch:04d}_weights.json", model)\n        atomic_json(out / "history.json", history)\n        atomic_json(out / "last.json", {"checkpoint": f"epoch_{epoch:04d}.pt"})\n        atomic_json(out / "best_weights.json", best_model)\n        patch = copy.deepcopy(config)\n        patch["weight_model"] = {"enabled": True, "path": str((out / "best_weights.json").resolve())}\n        (out / "best_partwise.yml").write_text(yaml.safe_dump(\n            {"fusion": {"method": "partwise", "partwise": patch}}, sort_keys=False), encoding="utf-8")\n        print(json.dumps(history[-1]), flush=True)\n    return history\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--manifest", required=True)\n    parser.add_argument("--cache", default="inputs/v4_cache")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--lr", type=float, default=0.01)\n    parser.add_argument("--resume")\n    parser.add_argument("--prepare-only", action="store_true")\n    parser.add_argument("--skip-unaligned-actions", action="store_true",\n                        help="Exclude sequences without usable camera pairs and record them in alignment.json")\n    args = parser.parse_args()\n    if args.epochs < 1 or args.lr <= 0:\n        parser.error("epochs and lr must be positive")\n    rows = json.loads(Path(args.manifest).read_text(encoding="utf-8"))\n    validate_rows(rows)\n    config = yaml.safe_load((ROOT / "configs/pipeline.yml").read_text(encoding="utf-8"))["fusion"]["partwise"]\n    config.pop("weight_model", None)\n    # Sheet lacks visibility/tracking results: match training and inference evidence.\n    config["multi_person"] = {"enabled": False}\n    from google.auth import default\n    from googleapiclient.discovery import build\n    credentials, _ = default()\n    drive = build("drive", "v3", credentials=credentials, cache_discovery=False)\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    cache = Path(args.cache)\n    identity = hashlib.sha256(Path(args.manifest).read_bytes() +\n        (ROOT / "models/J_regressor_body25_extended.npy").read_bytes() +\n        Path(__file__).read_bytes() + (ROOT / "fusion_pipeline/partwise.py").read_bytes() +\n        (ROOT / "configs/keypoints3D_map.yml").read_bytes() + json.dumps(config, sort_keys=True).encode() +\n        str(args.skip_unaligned_actions).encode()).hexdigest()\n    prepared = cache / f"prepared_{identity}.joblib"\n    if prepared.exists():\n        pairs, reports = joblib.load(prepared)\n    else:\n        pairs, reports = prepare(rows, drive, cache, config, out / "alignment.json",\n                                 skip_unaligned_actions=args.skip_unaligned_actions)\n        cache.mkdir(parents=True, exist_ok=True)\n        temp = prepared.with_suffix(".tmp")\n        joblib.dump((pairs, reports), temp)\n        temp.replace(prepared)\n    atomic_json(out / "alignment.json", reports)\n    atomic_json(out / "manifest.json", rows)\n    atomic_json(out / "protocol.json", {\n        "metric": "12-body-joint root-centered MPJPE, one GT rotation per camera-pair segment",\n        "selection": "mean of pair scores within each action, then mean across val actions",\n        "association": "GT-assisted proper-rotation PA timeline search and penalized track stitching; ratio 0.5/1/2, integer offset",\n        "limitations": "GT-assisted time/track association; review alignment.json. No independent test score.",\n        "feature_names": list(FEATURE_NAMES), "config": config,\n        "train_actions": sorted({p["action"] for p in pairs if p["split"] == "train"}),\n        "val_actions": sorted({p["action"] for p in pairs if p["split"] == "val"}),\n        "skip_unaligned_actions": args.skip_unaligned_actions,\n        "skipped_actions": [r["action"] for r in reports if r.get("status") == "skipped_action"],\n        "synchronization": "linear interpolation to shared GT ticks, excluding missing-track gaps",\n        "directed_segments": len(pairs),\n    })\n    if args.prepare_only:\n        print("Prepared data and alignment report:", out / "alignment.json")\n        return\n    fit(pairs, config, out, args.epochs, args.lr, args.resume, identity)\n\n\nif __name__ == "__main__":\n    main()\n', 'fusion_pipeline/partwise.py': '"""Part-wise symmetric two-view fusion (calibration-free, GT-free).\n\nFor every bone, the Master and Slave directions are blended and the skeleton is rebuilt\nby forward kinematics with the Master bone lengths (so MBLE equals the Master/baseline):\n\n    u = normalize(w * u_M + (1 - w) * R_part @ u_S)\n\n* R_part maps the Slave\'s torso part onto the Master\'s, with no extrinsics:\n  R_part = B_M @ B_S^T, where B is an orthonormal body frame built from\n  - lower part: hip axis + spine (pelvis->neck), used for hips/legs/feet;\n  - upper part: shoulder axis + spine, used for head/shoulders/arms/hands.\n  A single whole-body rotation twists the upper body when one view is poor; anchoring\n  each limb on the torso part it hangs from avoids that. R_part is smoothed over time.\n* w is the Master weight per frame: a logistic function of the root-depth difference\n  (depth_M - depth_S, camera frame). The farther view tends to be the more accurate\n  one on the design split (frame AUC 0.825; 0.863 on held-out). The logistic parameters\n  were fitted once on the design split and are frozen in the config.\n* A sequence-stability pass (fusion_pipeline.aotr_stability) then shrinks the\n  correction residual F - M where it would add acceleration.\n\nPelvis and neck stay at the Master positions. Only camera1 (Master) is modified.\n"""\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nfrom fusion_pipeline.adaptive_naive import bone_confidence, bone_jitter\nfrom fusion_pipeline.aotr import _ordered_bones\nfrom fusion_pipeline.detector import compute_bone_distortion_score\n\nLOWER_PART = frozenset({\n    "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle",\n    "left_foot", "right_foot", "left_toe", "right_toe",\n})\nREQUIRED = ("pelvis", "neck", "left_hip", "right_hip", "left_shoulder", "right_shoulder")\n\n\ndef _as_pose(frame):\n    return {k: np.asarray(v, dtype=float) for k, v in frame.items()}\n\n\ndef _orthonormalize(R):\n    U, _, Vt = np.linalg.svd(R)\n    D = np.eye(3)\n    D[2, 2] = np.sign(np.linalg.det(U @ Vt))\n    return U @ D @ Vt\n\n\ndef part_frame(pose, left, right):\n    """Orthonormal frame (columns x, y, z): x = right->left joint, y = spine, z = x × y."""\n    x = pose[left] - pose[right]\n    x = x / np.linalg.norm(x)\n    y = pose["neck"] - pose["pelvis"]\n    y = y - x * (x @ y)\n    y = y / np.linalg.norm(y)\n    return np.stack([x, y, np.cross(x, y)], axis=1)\n\n\ndef root_depth(pose):\n    return float((0.5 * (pose["left_hip"] + pose["right_hip"]))[2])\n\n\nLOWER_TORSO_JOINTS = ("pelvis", "neck", "left_hip", "right_hip")\nUPPER_TORSO_JOINTS = ("pelvis", "neck", "left_shoulder", "right_shoulder")\n\n\ndef _depth_logits(master, slave, logit):\n    d = np.array([root_depth(m) - root_depth(s) for m, s in zip(master, slave)], dtype=float)\n    d = np.nan_to_num(d)\n    return float(logit["a"]) + float(logit["b"]) * (d - float(logit["mu"])) / float(logit["sd"])\n\n\ndef _sigmoid(z):\n    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))\n\n\ndef master_weights(master, slave, logit):\n    """Per-frame Master weight from the root-depth difference (farther view favoured)."""\n    return _sigmoid(_depth_logits(master, slave, logit))\n\n\ndef _part_vis_weight(vis_m, vis_s, joints):\n    if not vis_m and not vis_s:\n        return 1.0\n    vm = vis_m or {}\n    vs = vis_s or {}\n    both = sum(bool(vm.get(j, True)) and bool(vs.get(j, True)) for j in joints)\n    return float(both) / float(len(joints))\n\n\nPRIMARY_LIMB_JOINTS = frozenset({\n    "left_elbow", "right_elbow", "left_wrist", "right_wrist",\n    "left_knee", "right_knee", "left_ankle", "right_ankle",\n})\n\n\ndef _median_bone_disagreement_deg(master_pose, slave_dirs, vis_m=None, vis_s=None):\n    """Zero-GT cross-view pose disagreement: median angle (deg) across primary limb bones."""\n    vm, vs = vis_m or {}, vis_s or {}\n    vis_angles, all_angles = [], []\n    for child, parent in _ordered_bones(master_pose):\n        if child not in PRIMARY_LIMB_JOINTS:\n            continue\n        us = slave_dirs.get(child)\n        if us is None:\n            continue\n        vec = master_pose[child] - master_pose[parent]\n        n = float(np.linalg.norm(vec))\n        if n <= 1e-9:\n            continue\n        ang = float(np.degrees(np.arccos(np.clip(float((vec / n) @ us), -1.0, 1.0))))\n        all_angles.append(ang)\n        if vm.get(child, True) and vs.get(child, True):\n            vis_angles.append(ang)\n    sample = vis_angles if len(vis_angles) >= 3 else all_angles\n    return float(np.median(sample)) if sample else 0.0\n\n\ndef _frame_bone_jump_deg(prev_pose, curr_pose):\n    if prev_pose is None:\n        return 0.0\n    angles = []\n    for child, parent in _ordered_bones(curr_pose):\n        v0 = prev_pose[child] - prev_pose[parent]\n        v1 = curr_pose[child] - curr_pose[parent]\n        n0, n1 = float(np.linalg.norm(v0)), float(np.linalg.norm(v1))\n        if n0 > 1e-9 and n1 > 1e-9:\n            angles.append(float(np.degrees(np.arccos(np.clip(float((v0 / n0) @ (v1 / n1)), -1.0, 1.0)))))\n    return float(np.median(angles)) if angles else 0.0\n\n\ndef slave_directions(master, slave, half_window=4, vis_master=None, vis_slave=None, frame_weights=None):\n    """Slave bone unit vectors in the Master frame, one dict per frame."""\n    n_frames = len(master)\n    rot = {"lower": [], "upper": []}\n    part_w = {"lower": np.ones(n_frames, dtype=float), "upper": np.ones(n_frames, dtype=float)}\n    for t, (m, s) in enumerate(zip(master, slave)):\n        rot["lower"].append(part_frame(m, "left_hip", "right_hip") @ part_frame(s, "left_hip", "right_hip").T)\n        rot["upper"].append(part_frame(m, "left_shoulder", "right_shoulder")\n                            @ part_frame(s, "left_shoulder", "right_shoulder").T)\n        base_fw = 1.0 if frame_weights is None else float(frame_weights[t])\n        vm = vis_master[t] if vis_master is not None else None\n        vs = vis_slave[t] if vis_slave is not None else None\n        part_w["lower"][t] = base_fw * _part_vis_weight(vm, vs, LOWER_TORSO_JOINTS)\n        part_w["upper"][t] = base_fw * _part_vis_weight(vm, vs, UPPER_TORSO_JOINTS)\n\n    # ponytail: precompute nearest clean frame fallback when an entire local window is occluded\n    clean_idx = {k: np.where(part_w[k] > 0.5)[0] for k in ("lower", "upper")}\n    bones = _ordered_bones(master[0])\n    out = []\n    for t, s in enumerate(slave):\n        lo, hi = max(0, t - half_window), min(n_frames, t + half_window + 1)\n        R = {}\n        for k in ("lower", "upper"):\n            weights = part_w[k][lo:hi]\n            w_sum = float(np.sum(weights))\n            if w_sum > 1e-6:\n                R_avg = sum(w_i * R_i for w_i, R_i in zip(weights, rot[k][lo:hi])) / w_sum\n                R[k] = _orthonormalize(R_avg)\n            elif len(clean_idx[k]) > 0:\n                nearest = int(clean_idx[k][np.argmin(np.abs(clean_idx[k] - t))])\n                R[k] = _orthonormalize(rot[k][nearest])\n            else:\n                R[k] = _orthonormalize(np.mean(rot[k][lo:hi], axis=0))\n        dirs = {}\n        for child, parent in bones:\n            v = s[child] - s[parent]\n            n = np.linalg.norm(v)\n            if n > 1e-9:\n                dirs[child] = R["lower" if child in LOWER_PART else "upper"] @ (v / n)\n        out.append(dirs)\n    return out\n\n\ndef blend_frame(master_pose, slave_dirs, w, bone_weights=None):\n    """Blend bone directions (w = Master weight, or per-bone dict) and rebuild with Master bone lengths."""\n    out = {k: v.copy() for k, v in master_pose.items()}\n    for child, parent in _ordered_bones(master_pose):\n        um = master_pose[child] - master_pose[parent]\n        length = np.linalg.norm(um)\n        if length < 1e-9:\n            out[child] = out[parent] + (master_pose[child] - master_pose[parent])\n            continue\n        um = um / length\n        us = slave_dirs.get(child)\n        wb = float(bone_weights[child]) if (bone_weights is not None and child in bone_weights) else float(w)\n        u = um if us is None else wb * um + (1.0 - wb) * us\n        nu = np.linalg.norm(u)\n        u = um if nu < 1e-9 else u / nu\n        out[child] = out[parent] + length * u\n    return out\n\n\ndef _geodesic_deg(R1, R2):\n    tr = float(np.trace(R1.T @ R2))\n    return float(np.degrees(np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))))\n\n\ndef _torso_frame(pose):\n    return _orthonormalize(0.5 * (\n        part_frame(pose, "left_hip", "right_hip")\n        + part_frame(pose, "left_shoulder", "right_shoulder")\n    ))\n\n\ndef estimate_static_camera_rotation(master, slave, vis_master=None, vis_slave=None,\n                                    frame_valid=None, inlier_deg=25.0):\n    """Zero-GT robust SO(3) medoid + inlier mean for ONE static camera rotation R_cam (Slave -> Master)."""\n    cands, weights = [], []\n    for t, (m, s) in enumerate(zip(master, slave)):\n        if frame_valid is not None and frame_valid[t] <= 0.0:\n            continue\n        vm = vis_master[t] if vis_master is not None else None\n        vs = vis_slave[t] if vis_slave is not None else None\n        wl = _part_vis_weight(vm, vs, LOWER_TORSO_JOINTS)\n        wu = _part_vis_weight(vm, vs, UPPER_TORSO_JOINTS)\n        if wl >= 0.5:\n            cands.append(part_frame(m, "left_hip", "right_hip") @ part_frame(s, "left_hip", "right_hip").T)\n            weights.append(wl)\n        if wu >= 0.5:\n            cands.append(part_frame(m, "left_shoulder", "right_shoulder") @ part_frame(s, "left_shoulder", "right_shoulder").T)\n            weights.append(wu)\n    if not cands:\n        for m, s in zip(master, slave):\n            cands.append(_torso_frame(m) @ _torso_frame(s).T)\n            weights.append(1.0)\n    step = max(1, len(cands) // 60)\n    best_R, best_med = cands[0], float("inf")\n    for idx in range(0, len(cands), step):\n        R0 = cands[idx]\n        med = float(np.median([_geodesic_deg(R0, R) for R in cands]))\n        if med < best_med:\n            best_med, best_R = med, R0\n    inliers = [(w, R) for w, R in zip(weights, cands) if _geodesic_deg(best_R, R) <= float(inlier_deg)]\n    if not inliers:\n        return _orthonormalize(best_R), best_med\n    w_sum = sum(w for w, _ in inliers)\n    return _orthonormalize(sum(w * R for w, R in inliers) / max(w_sum, 1e-9)), best_med\n\n\ndef _apply_static_global_orientation(fused_lists, M, S, z_depth, vis_master=None, vis_slave=None,\n                                     frame_valid=None, smooth_hw=2):\n    """Decoupled global-orientation fusion for static cameras.\n\n    Applies ONE rigid rotation dR(t) per frame about pelvis so PA-MPJPE and MBLE are\n    mathematically invariant while sequence-aligned MPJPE gains from static R_cam.\n    """\n    R_cam, med_deg = estimate_static_camera_rotation(M, S, vis_master, vis_slave, frame_valid=frame_valid)\n    # ponytail: fail-closed when cross-view R_cam is noisy (e.g. broken Master on MMA/grappling)\n    if med_deg > 10.0:\n        return fused_lists, med_deg\n    bm = [_torso_frame(m) for m in M]\n    bs = [R_cam @ _torso_frame(s) for s in S]\n    torso_j = ("pelvis", "neck", "left_hip", "right_hip", "left_shoulder", "right_shoulder")\n\n    targets = []\n    for t, zt in enumerate(z_depth):\n        vm = vis_master[t] if vis_master is not None else {}\n        vs = vis_slave[t] if vis_slave is not None else {}\n        occ_m = sum(not vm.get(j, True) for j in torso_j)\n        occ_s = sum(not vs.get(j, True) for j in torso_j)\n        zt_eff = float(zt) + 0.6 * (occ_s - occ_m)\n        ang = _geodesic_deg(bm[t], bs[t])\n        if ang > 45.0:\n            jm = _geodesic_deg(bm[t - 1], bm[t]) if t > 0 else 0.0\n            js = _geodesic_deg(bs[t - 1], bs[t]) if t > 0 else 0.0\n            if jm > max(15.0, 2.0 * js):\n                zt_eff = min(zt_eff, 0.0) - 3.0\n            elif js > max(15.0, 2.0 * jm):\n                zt_eff = max(zt_eff, 0.0) + 3.0\n        w_ori = float(_sigmoid(zt_eff))\n        if ang > 140.0:\n            B_tgt = bm[t] if w_ori >= 0.5 else bs[t]\n        else:\n            B_tgt = _orthonormalize(w_ori * bm[t] + (1.0 - w_ori) * bs[t])\n        targets.append(B_tgt)\n\n    # Smooth target orientation locally (skip averaging across >45 deg discontinuities)\n    n = len(targets)\n    out = []\n    for t, pose_dict in enumerate(fused_lists):\n        if smooth_hw > 0 and n > 2:\n            lo, hi = max(0, t - smooth_hw), min(n, t + smooth_hw + 1)\n            local = [targets[k] for k in range(lo, hi) if _geodesic_deg(targets[t], targets[k]) <= 45.0]\n            B_smooth = _orthonormalize(np.mean(local, axis=0)) if local else targets[t]\n        else:\n            B_smooth = targets[t]\n        pose_arr = {k: np.asarray(v, dtype=float) for k, v in pose_dict.items()}\n        # Rotate from the fused skeleton\'s ACTUAL torso frame to B_smooth (avoids double-rotating\n        # when windowed R_part already shifted hips/shoulders toward Slave).\n        dR = B_smooth @ _torso_frame(pose_arr).T\n        root = pose_arr["pelvis"]\n        out.append({k: (root + dR @ (v - root)).tolist() for k, v in pose_arr.items()})\n    return out, med_deg\n\n\nFEATURE_NAMES = (\n    "d_depth", "d_occ_joint", "d_occ_torso", "d_conf2d",\n    "d_bone_jit", "d_cam_accel", "d_bone_distort", "d_facing",\n)\n\n\ndef _facing_abs_cos(pose):\n    """|cos(angle)| between torso normal and camera optical ray -> 1 = frontal, 0 = side profile."""\n    x = pose["left_shoulder"] - pose["right_shoulder"]\n    y = pose["neck"] - pose["pelvis"]\n    normal = np.cross(x, y)\n    nn = float(np.linalg.norm(normal))\n    ray = 0.5 * (pose["left_hip"] + pose["right_hip"])\n    rn = float(np.linalg.norm(ray))\n    if nn <= 1e-9 or rn <= 1e-9:\n        return 0.5\n    return float(abs((normal / nn) @ (ray / rn)))\n\n\ndef _seq_median_accel_mm(poses):\n    if len(poses) < 3:\n        return 0.0\n    keys = sorted(poses[0])\n    arr = np.array([[p[k] for k in keys] for p in poses], dtype=float)\n    acc = np.linalg.norm(arr[2:] - 2.0 * arr[1:-1] + arr[:-2], axis=-1)\n    return float(np.median(acc[5:] if len(acc) > 8 else acc) * 1000.0)\n\n\ndef extract_pair_features(M, S, vis_master=None, vis_slave=None, conf_master=None, conf_slave=None):\n    """Return (features[T, B, 8], bone_children) with antisymmetric features phi(M, S) = -phi(S, M)."""\n    n = len(M)\n    bones = _ordered_bones(M[0])\n    b_children = [c for c, _ in bones]\n    d_depth = np.clip(np.array([root_depth(m) - root_depth(s) for m, s in zip(M, S)], dtype=float), -3.0, 3.0)\n    jm, js = bone_jitter(M, bones, 3), bone_jitter(S, bones, 3)\n    j_med = {}\n    for c in b_children:\n        pooled = np.concatenate([jm[c], js[c]])\n        pooled = pooled[np.isfinite(pooled)]\n        j_med[c] = float(np.median(pooled)) if pooled.size and float(np.median(pooled)) > 1e-12 else 1.0\n    acc_m, acc_s = _seq_median_accel_mm(M), _seq_median_accel_mm(S)\n    d_cam_acc = float(np.clip(np.log((acc_s + 1.0) / (acc_m + 1.0)), -3.0, 3.0))\n\n    feats = np.zeros((n, len(bones), len(FEATURE_NAMES)), dtype=np.float32)\n    for t in range(n):\n        vm = (vis_master[t] if vis_master else None) or {}\n        vs = (vis_slave[t] if vis_slave else None) or {}\n        cm_t = conf_master[t] if conf_master else None\n        cs_t = conf_slave[t] if conf_slave else None\n        occ_m = float(sum(not vm.get(j, True) for j in M[t])) / max(len(M[t]), 1)\n        occ_s = float(sum(not vs.get(j, True) for j in S[t])) / max(len(S[t]), 1)\n        d_occ_torso = occ_s - occ_m\n        d_distort = float(np.clip(compute_bone_distortion_score(S[t]) - compute_bone_distortion_score(M[t]), -1.0, 1.0))\n        d_face = _facing_abs_cos(M[t]) - _facing_abs_cos(S[t])\n        for bi, (child, parent) in enumerate(bones):\n            d_occ_j = float(bool(vm.get(child, True))) - float(bool(vs.get(child, True)))\n            c_m = bone_confidence(cm_t, child, parent, 0.5)\n            c_s = bone_confidence(cs_t, child, parent, 0.5)\n            d_c2d = float(np.log((c_m + 0.05) / (c_s + 0.05)))\n            nj_m = float(jm[child][t] / j_med[child]) if np.isfinite(jm[child][t]) else 0.0\n            nj_s = float(js[child][t] / j_med[child]) if np.isfinite(js[child][t]) else 0.0\n            d_jit = float(np.clip(np.log((nj_s + 0.1) / (nj_m + 0.1)), -3.0, 3.0))\n            feats[t, bi] = (d_depth[t], d_occ_j, d_occ_torso, d_c2d, d_jit, d_cam_acc, d_distort, d_face)\n    return feats, b_children\n\n\ndef eval_weight_model(feats, wm_cfg):\n    """Evaluate learned antisymmetric weight model -> w_Master[T, B] in (0, 1)."""\n    if isinstance(wm_cfg.get("path"), str) and wm_cfg["path"].strip():\n        p = Path(wm_cfg["path"])\n        if not p.is_absolute():\n            p = Path(__file__).resolve().parents[1] / p\n        wm_cfg = {**json.loads(p.read_text(encoding="utf-8")), **wm_cfg}\n    mean = np.asarray(wm_cfg.get("mean", [0.0] * feats.shape[-1]), dtype=float)\n    std = np.maximum(np.asarray(wm_cfg.get("std", [1.0] * feats.shape[-1]), dtype=float), 1e-6)\n    # Center-only or symmetric scale: to preserve phi(M,S) = -phi(S,M), mean must be 0!\n    x = (feats - mean) / std\n    kind = str(wm_cfg.get("type", "linear")).lower()\n    if kind == "sym_mlp":\n        W1 = np.asarray(wm_cfg["W1"], dtype=float)\n        b1 = np.asarray(wm_cfg["b1"], dtype=float)\n        W2 = np.asarray(wm_cfg["W2"], dtype=float).reshape(-1)\n        logits = 0.5 * (np.tanh(x @ W1 + b1) @ W2 - np.tanh((-x) @ W1 + b1) @ W2)\n    else:\n        w_vec = np.asarray(wm_cfg["weights"], dtype=float).reshape(-1)\n        logits = x @ w_vec\n    return _sigmoid(logits)\n\n\ndef fuse_partwise(master_frames, slave_frames, config, vis_master=None, vis_slave=None,\n                  conf_master=None, conf_slave=None):\n    """Return (camera1 frames as lists, diagnostics). Inputs: per-frame {joint: xyz}."""\n    if len(master_frames) != len(slave_frames) or not master_frames:\n        raise ValueError("partwise fusion needs equally long, non-empty Master/Slave sequences")\n    M = [_as_pose(f) for f in master_frames]\n    S = [_as_pose(f) for f in slave_frames]\n    missing = [j for j in REQUIRED if j not in M[0] or j not in S[0]]\n    if missing:\n        raise ValueError(f"partwise fusion requires joints {missing}")\n    z_m_arr = np.array([root_depth(m) for m in M], dtype=float)\n    z_s_arr = np.array([root_depth(s) for s in S], dtype=float)\n    z_depth = _depth_logits(M, S, config["depth_logit"])\n    w = _sigmoid(z_depth)\n    hw = int(config.get("rotation_half_window", 4))\n    mp_cfg = dict(config.get("multi_person", {}) or {})\n    mp_enabled = bool(mp_cfg.get("enabled", False))\n\n    swap_gated = 0\n    depth_outlier_frames = 0\n    occ_master_bones = 0\n    occ_slave_bones = 0\n    disagreements = []\n\n    wm_cfg = config.get("weight_model")\n    use_wm = isinstance(wm_cfg, dict) and bool(wm_cfg.get("enabled", True))\n    if use_wm:\n        feats, b_children = extract_pair_features(M, S, vis_master, vis_slave, conf_master, conf_slave)\n        w_mat = eval_weight_model(feats, wm_cfg)\n        use_vis_rot = bool(mp_cfg.get("visibility_weighted_rotation", True)) if mp_enabled else False\n        anti_swap = bool(mp_cfg.get("anti_swap_gate", True)) if mp_enabled else False\n        max_ang = float(mp_cfg.get("max_median_bone_angle_deg", 50.0))\n        dirs_init = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n        )\n        frame_valid = np.ones(len(M), dtype=float)\n        frame_gate_keep = [False] * len(M)\n        for t in range(len(M)):\n            vm = (vis_master[t] if vis_master is not None else None) or {}\n            vs = (vis_slave[t] if vis_slave is not None else None) or {}\n            ang = _median_bone_disagreement_deg(M[t], dirs_init[t], vm, vs)\n            disagreements.append(ang)\n            if anti_swap and ang > max_ang:\n                frame_valid[t] = 0.0\n                jump_m = _frame_bone_jump_deg(M[t - 1] if t > 0 else None, M[t])\n                jump_s = _frame_bone_jump_deg(S[t - 1] if t > 0 else None, S[t])\n                occ_m_ratio = float(sum(not vm.get(j, True) for j in M[t])) / max(len(M[t]), 1)\n                if not (occ_m_ratio >= 0.35 and jump_m > max(15.0, 2.0 * jump_s)):\n                    frame_gate_keep[t] = True\n                    swap_gated += 1\n        dirs = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n            frame_weights=frame_valid if anti_swap else None,\n        ) if anti_swap and np.any(frame_valid == 0.0) else dirs_init\n        fused = []\n        for t, (m, d) in enumerate(zip(M, dirs)):\n            if frame_gate_keep[t]:\n                fused.append({k: v.copy() for k, v in m.items()})\n            else:\n                bw = {child: float(w_mat[t, bi]) for bi, child in enumerate(b_children)}\n                fused.append(blend_frame(m, d, float(np.mean(w_mat[t])), bone_weights=bw))\n        w = np.mean(w_mat, axis=1)\n    elif mp_enabled:\n        # ponytail: guard against occlusion-induced bbox-collapse depth explosion (z -> 2x normal)\n        # In monocular HMR, partial occlusion shrinks bbox -> z spikes to 12-20m -> raw depth_logit\n        # would falsely assign w=0.0 to a clean Master and 100% to the collapsed Slave.\n        max_depth_diff = float(mp_cfg.get("max_depth_diff_m", 3.0))\n        depth_outlier_ratio = float(mp_cfg.get("depth_outlier_ratio", 1.5))\n        depth_outlier_min_m = float(mp_cfg.get("depth_outlier_min_m", 10.0))\n        depth_outlier_bias = float(mp_cfg.get("depth_outlier_logit_bias", 3.0))\n        z_m_med = float(np.median(z_m_arr[np.isfinite(z_m_arr)])) if np.any(np.isfinite(z_m_arr)) else 5.0\n        z_s_med = float(np.median(z_s_arr[np.isfinite(z_s_arr)])) if np.any(np.isfinite(z_s_arr)) else 5.0\n        d_clamped = np.clip(np.nan_to_num(z_m_arr - z_s_arr), -max_depth_diff, max_depth_diff)\n        logit = config["depth_logit"]\n        z_depth = float(logit["a"]) + float(logit["b"]) * (d_clamped - float(logit["mu"])) / float(logit["sd"])\n        for t in range(len(M)):\n            zm, zs = float(z_m_arr[t]), float(z_s_arr[t])\n            s_exploded = (zs >= depth_outlier_min_m and (zs > depth_outlier_ratio * max(zm, 1e-3) or zs > depth_outlier_ratio * max(z_s_med, 1e-3)))\n            m_exploded = (zm >= depth_outlier_min_m and (zm > depth_outlier_ratio * max(zs, 1e-3) or zm > depth_outlier_ratio * max(z_m_med, 1e-3)))\n            if s_exploded and not m_exploded:\n                z_depth[t] = max(z_depth[t], 0.0) + depth_outlier_bias\n                depth_outlier_frames += 1\n            elif m_exploded and not s_exploded:\n                z_depth[t] = min(z_depth[t], 0.0) - depth_outlier_bias\n                depth_outlier_frames += 1\n        w = _sigmoid(z_depth)\n        use_vis_rot = bool(mp_cfg.get("visibility_weighted_rotation", True))\n        dirs_init = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n        )\n        anti_swap = bool(mp_cfg.get("anti_swap_gate", True))\n        max_ang = float(mp_cfg.get("max_median_bone_angle_deg", 50.0))\n        frame_valid = np.ones(len(M), dtype=float)\n        frame_gate_keep = [False] * len(M)\n        for t in range(len(M)):\n            vm = vis_master[t] if vis_master is not None else {}\n            vs = vis_slave[t] if vis_slave is not None else {}\n            ang = _median_bone_disagreement_deg(M[t], dirs_init[t], vm, vs)\n            disagreements.append(ang)\n            if anti_swap and ang > max_ang:\n                frame_valid[t] = 0.0\n                jump_m = _frame_bone_jump_deg(M[t - 1] if t > 0 else None, M[t])\n                jump_s = _frame_bone_jump_deg(S[t - 1] if t > 0 else None, S[t])\n                occ_m_ratio = float(sum(not vm.get(j, True) for j in M[t])) / max(len(M[t]), 1)\n                # Keep Master untouched unless Master itself clearly broke/jumped under heavy occlusion\n                if not (occ_m_ratio >= 0.35 and jump_m > max(15.0, 2.0 * jump_s)):\n                    frame_gate_keep[t] = True\n                    swap_gated += 1\n        # Recompute R_part excluding suspected swap frames so they cannot contaminate neighbors\n        dirs = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n            frame_weights=frame_valid if anti_swap else None,\n        ) if anti_swap and np.any(frame_valid == 0.0) else dirs_init\n\n        m_occ_bias = float(mp_cfg.get("master_occluded_logit_bias", 2.0))\n        s_occ_bias = float(mp_cfg.get("slave_occluded_logit_bias", 2.5))\n        both_occ_bias = float(mp_cfg.get("both_occluded_logit_bias", 0.5))\n        bones = _ordered_bones(M[0])\n        fused = []\n        for t, (m, d, zt, wt) in enumerate(zip(M, dirs, z_depth, w)):\n            if frame_gate_keep[t]:\n                fused.append({k: v.copy() for k, v in m.items()})\n                continue\n            vm = vis_master[t] if vis_master is not None else None\n            vs = vis_slave[t] if vis_slave is not None else None\n            if vm is None and vs is None:\n                fused.append(blend_frame(m, d, wt))\n                continue\n            vm, vs = vm or {}, vs or {}\n            bw = {}\n            for child, _ in bones:\n                m_vis = bool(vm.get(child, True))\n                s_vis = bool(vs.get(child, True))\n                if not m_vis and s_vis:\n                    # Occluded Master + visible Slave: never let depth prior override visibility\n                    bw[child] = float(_sigmoid(min(zt, 0.0) - m_occ_bias))\n                    occ_master_bones += 1\n                elif m_vis and not s_vis:\n                    # Visible Master + occluded Slave: protect Master regardless of depth\n                    bw[child] = float(_sigmoid(max(zt, 0.0) + s_occ_bias))\n                    occ_slave_bones += 1\n                elif not m_vis and not s_vis:\n                    bw[child] = float(_sigmoid(zt + both_occ_bias))\n                else:\n                    bw[child] = float(wt)\n            fused.append(blend_frame(m, d, wt, bone_weights=bw))\n    else:\n        dirs = slave_directions(M, S, hw)\n        fused = [blend_frame(m, d, wt) for m, d, wt in zip(M, dirs, w)]\n\n    diag = {\n        "mode": "partwise",\n        "frames": len(fused),\n        "master_weight_mean": float(np.mean(w)),\n        "master_weight_min": float(np.min(w)),\n        "master_weight_max": float(np.max(w)),\n        "multi_person_enabled": mp_enabled,\n        "swap_gated_frames": swap_gated,\n        "depth_outlier_frames": depth_outlier_frames,\n        "occluded_master_bones": occ_master_bones,\n        "occluded_slave_bones": occ_slave_bones,\n        "median_bone_angle_deg_mean": float(np.mean(disagreements)) if disagreements else None,\n        "sequence_stability": {"status": "disabled"},\n    }\n    as_lists = [{k: v.tolist() for k, v in p.items()} for p in fused]\n    stab_cfg = dict(config.get("sequence_stability", {}))\n    if stab_cfg.get("enabled", True):\n        from fusion_pipeline.aotr_stability import postprocess_aotr_3d\n        master_lists = [{k: v.tolist() for k, v in p.items()} for p in M]\n        as_lists, stab_diag = postprocess_aotr_3d(\n            master_lists, as_lists, config=stab_cfg, editable_mask=[True] * len(as_lists))\n        diag["sequence_stability"] = stab_diag\n    if mp_enabled and bool(mp_cfg.get("static_camera_orientation", False)):\n        as_lists, rcam_med = _apply_static_global_orientation(\n            as_lists, M, S, z_depth,\n            vis_master=vis_master, vis_slave=vis_slave,\n            frame_valid=frame_valid if anti_swap else None,\n            smooth_hw=int(mp_cfg.get("orientation_smooth_half_window", 2)),\n        )\n        diag["static_camera_orientation"] = {"enabled": True, "r_cam_median_deg": round(rcam_med, 2)}\n    return as_lists, diag\n', 'tests/test_train_v4.py': 'import copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport joblib\nimport numpy as np\nimport torch\n\nfrom train_v4 import (AlignmentError, associate, centered, export_model, fit, prepare,\n                      torch_fuse, validate_rows, synchronize)\nfrom keypoints_map import load_keypoints3d_map\nfrom fusion_pipeline.partwise import extract_pair_features, fuse_partwise, slave_directions\nfrom test_partwise import CFG, skeleton\n\n\ndef pair(split="train"):\n    master = [skeleton(elbow_lift=0.01 * t) for t in range(10)]\n    slave = [skeleton(depth=6, elbow_lift=0.02 * t) for t in range(10)]\n    names = list(master[0])\n    features, children = extract_pair_features(master, slave)\n    return dict(action=split, split=split, M=master, S=slave, names=names,\n                features=features, children=children,\n                directions=slave_directions(master, slave, 4),\n                target=centered(np.array([[f[n] for n in names] for f in slave]), names).astype(np.float32))\n\n\nclass TrainingTests(unittest.TestCase):\n    def test_prepare_reports_rejected_camera_and_keeps_valid_pairs(self):\n        mapping = load_keypoints3d_map("configs/keypoints3D_map.yml")\n        names = [k["name"] for k in mapping["keypoints"]]\n        joints = np.array([[skeleton(elbow_lift=0.01*t)[n] for n in names] for t in range(24)])\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            rows = []\n            for split, count in (("train", 3), ("val", 2)):\n                gt_id = f"{split}_ground_truth"\n                with (root / gt_id).open("wb") as stream:\n                    np.savez(stream, verts=joints)\n                for i in range(count):\n                    pkl_id = f"{split}_camera_{i}"\n                    length = 8 if split == "train" and i == 0 else len(joints)\n                    joblib.dump({"0": dict(frame_ids=np.arange(length), verts=joints[:length])}, root / pkl_id)\n                    rows.append(dict(action=split, split=split, cam=f"cam{i}",\n                                     GT=gt_id, **{"file PKL": pkl_id}))\n            report_path = root / "alignment.json"\n            with patch("train_v4.download", side_effect=lambda drive, value, cache: root/value), \\\n                 patch("train_v4.joints_from_vertices", side_effect=lambda vertices, reg: vertices):\n                pairs, reports = prepare(rows, None, root, CFG, report_path)\n                rejected = [r for r in reports if r["status"] == "rejected"]\n                self.assertEqual([(r["action"], r["cam"]) for r in rejected], [("train", "cam0")])\n                self.assertEqual(json.loads(report_path.read_text()), reports)\n                self.assertEqual({p["split"] for p in pairs}, {"train", "val"})\n                self.assertTrue(all("cam0" not in p["cameras"] for p in pairs if p["split"] == "train"))\n                with self.assertRaisesRegex(ValueError, "train: need at least two aligned cameras"):\n                    prepare([r for r in rows if not (r["split"] == "train" and r["cam"] == "cam2")],\n                            None, root, CFG, report_path)\n                self.assertEqual(json.loads(report_path.read_text())[0]["status"], "rejected")\n                bad_rows = [dict(action="bad", split="train", cam=f"cam{i}",\n                                 GT="train_ground_truth", **{"file PKL": "train_camera_0"})\n                            for i in range(2)]\n                pairs, reports = prepare(rows + bad_rows, None, root, CFG, report_path,\n                                         skip_unaligned_actions=True)\n                self.assertEqual({p["action"] for p in pairs}, {"train", "val"})\n                skipped = [r for r in reports if r["status"] == "skipped_action"]\n                self.assertEqual([r["action"] for r in skipped], ["bad"])\n                self.assertEqual(json.loads(report_path.read_text()), reports)\n                self.assertTrue(all(r["camera_frames"] == 8 and r["gt_frames"] == 24\n                                    for r in reports if r["status"] == "rejected"))\n                with self.assertRaisesRegex(ValueError, "No usable pairs for one of the splits"):\n                    prepare(bad_rows + [r for r in rows if r["split"] == "val"], None, root, CFG,\n                            report_path, skip_unaligned_actions=True)\n\n    def test_alignment_rejects_insufficient_overlap(self):\n        names = pair()["names"]\n        gt = np.zeros((100, len(names), 3))\n        with self.assertRaisesRegex(AlignmentError, "No alignment covers"):\n            associate({"0": (np.arange(16), gt[:16])}, gt, names)\n\n    def test_leakage_and_duplicates_rejected(self):\n        rows = [dict(action=a, cam=c, split=s, GT=g, **{"file PKL": "some_id"})\n                for a,s,g in [("a","train","g1"),("b","val","g2")] for c in ["cam0","cam1"]]\n        validate_rows(rows)\n        bad = copy.deepcopy(rows)\n        bad[1]["split"] = "val"\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            validate_rows(bad)\n        with self.assertRaisesRegex(ValueError, "Duplicate"):\n            validate_rows(rows + rows[:1])\n        bad = copy.deepcopy(rows)\n        bad[2]["GT"] = "g1"\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            validate_rows(bad)\n\n    def test_differentiable_fusion_matches_inference(self):\n        p = pair()\n        w = torch.nn.Parameter(torch.arange(8, dtype=torch.float32) / 10)\n        std = torch.ones(8)\n        actual = torch_fuse(p, w, std)\n        cfg = dict(CFG, weight_model=export_model(w.detach().numpy(), std.numpy()))\n        result, _ = fuse_partwise(p["M"], p["S"], cfg)\n        expected = centered(np.array([[f[n] for n in p["names"]] for f in result]), p["names"])\n        np.testing.assert_allclose(actual.detach().numpy(), expected, atol=2e-6)\n        actual.square().sum().backward()\n        self.assertTrue(torch.isfinite(w.grad).all())\n\n    def test_association_recovers_track_and_offset(self):\n        rng = np.random.default_rng(9)\n        p = pair()\n        names = p["names"]\n        gt = rng.normal(size=(45, len(names), 3))\n        correct = gt[7:37] @ np.array([[0,-1,0],[1,0,0],[0,0,1]]) + 4\n        wrong = rng.normal(size=correct.shape)\n        ids, joints, report = associate({"wrong": (np.arange(30), wrong),\n                                        "right": (np.arange(30), correct)}, gt, names)\n        self.assertEqual(report["track"], "right")\n        self.assertEqual(report["ratio"], 1)\n        self.assertEqual(report["offset"], 7)\n        np.testing.assert_array_equal(ids, np.arange(7,37))\n\n    def test_fragmented_tracks_and_double_rate_gt(self):\n        rng = np.random.default_rng(2)\n        names = pair()["names"]\n        gt = rng.normal(size=(60, len(names), 3))\n        joints = gt[::2] + 4\n        ids, actual, report = associate({"first": (np.arange(15), joints[:15]),\n                                        "second": (np.arange(15,30), joints[15:])}, gt, names)\n        self.assertEqual(report["ratio"], 2)\n        self.assertEqual(report["offset"], 0)\n        self.assertEqual(report["switches"], 1)\n        np.testing.assert_array_equal(ids, np.arange(0,60,2))\n        np.testing.assert_allclose(actual, joints)\n\n    def test_camera_phase_interpolation(self):\n        im, is_ = np.arange(0,20,2), np.arange(1,20,2)\n        jm, js = im[:,None,None] * np.ones((1,3,3)), is_[:,None,None] * np.ones((1,3,3))\n        ticks, master, slave = synchronize(im,jm,is_,js)\n        np.testing.assert_array_equal(ticks, np.arange(1,19))\n        np.testing.assert_allclose(master, slave)\n        np.testing.assert_allclose(master[:,0,0], ticks)\n\n    def test_checkpoint_best_and_resume_match_uninterrupted(self):\n        pairs = [pair(), pair("val")]\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            full = fit(pairs, CFG, root/"full", epochs=3)\n            fit(pairs, CFG, root/"part", epochs=1)\n            resumed = fit(pairs, CFG, root/"resume", epochs=3, resume=root/"part/epoch_0001.pt")\n            self.assertEqual(full, resumed)\n            best = json.loads((root/"resume/best_weights.json").read_text())\n            self.assertEqual(best["val_mpjpe_mm"], min(h["val_mpjpe_mm"] for h in resumed))\n            self.assertTrue((root/"resume/epoch_0003.pt").exists())\n            for key in ["weights","std"]:\n                a = json.loads((root/"full/epoch_0003_weights.json").read_text())[key]\n                b = json.loads((root/"resume/epoch_0003_weights.json").read_text())[key]\n                np.testing.assert_allclose(a,b)\n            with self.assertRaisesRegex(ValueError, "differs"):\n                fit(pairs, CFG, root/"bad", epochs=3, lr=0.2, resume=root/"part/epoch_0001.pt")\n\n\nif __name__ == "__main__":\n    unittest.main()\n'}
for name, text in SNAPSHOTS.items():
    path = REPO_DIR/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests",
     "-p", "test_train_v4.py", "-v"], cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding="utf-8", errors="replace")
print(test_result.stdout, flush=True)
if test_result.returncode:
    raise RuntimeError("Trainer tests failed:\n" + test_result.stdout)


In [ ]:
# 3. Đăng nhập và nơi lưu checkpoint
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
import google.auth, gspread
credentials, _ = google.auth.default()
client = gspread.authorize(credentials)
SPREADSHEET_ID = "11OipPLik-aXw0GmrTfEwg6wsAQYEmvGEV8VRlyry4Rg"
TAB = "v4 22/11/26"
OUTPUT = Path("/content/drive/MyDrive/partwise_v4/run_001")
CACHE = Path("/content/v4_cache")
EPOCHS = 50
LEARNING_RATE = 0.01
RESUME = True
SKIP_UNALIGNED_ACTIONS = True  # Report and exclude sequences without valid camera pairs.
OUTPUT.mkdir(parents=True, exist_ok=True)

sheet = client.open_by_key(SPREADSHEET_ID).worksheet(TAB)
values = sheet.get(f"A1:O{sheet.row_count}")
headers = values[0]
rows = [dict(zip(headers, row + [""] * (len(headers)-len(row))))
        for row in values[1:] if any(row)]
from train_v4 import validate_rows
groups = validate_rows(rows)
print("Rows:", len(rows), "Train actions:", list(groups.values()).count("train"),
      "Val actions:", list(groups.values()).count("val"))
MANIFEST = OUTPUT/"manifest_input.json"
text = json.dumps(rows, ensure_ascii=False, indent=2)
# Keep the original input snapshot on resume; don't silently replace it.
if (OUTPUT/"last.json").exists() and MANIFEST.exists():
    assert json.loads(MANIFEST.read_text()) == rows, "Sheet changed: use a new OUTPUT folder"
MANIFEST.write_text(text, encoding="utf-8")


In [ ]:
# 4a. Prepare data and review alignment before training
prepare_command = [sys.executable, "-u", "train_v4.py", "--manifest", str(MANIFEST),
                   "--cache", str(CACHE), "--output", str(OUTPUT), "--prepare-only"]
if SKIP_UNALIGNED_ACTIONS:
    prepare_command.append("--skip-unaligned-actions")
prepare_log = []
with subprocess.Popen(prepare_command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                      errors="replace", bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        prepare_log.append(line)
    returncode = process.wait()
if returncode:
    raise RuntimeError("Data preparation failed:\n" + "".join(prepare_log))
import pandas as pd
alignment = pd.DataFrame(json.loads((OUTPUT/"alignment.json").read_text()))
display(alignment)
print("GT-assisted matching: review identity, timeline and coverage before training.")
display(alignment[alignment["status"].isin(["rejected", "skipped_action"]) |
                  (alignment["coverage"] < 0.8) | (alignment["pa_mm"] > 80)])


In [ ]:
# 4. Train / tiếp tục từ epoch cuối
command = [sys.executable, "-u", "train_v4.py", "--manifest", str(MANIFEST),
           "--cache", str(CACHE), "--output", str(OUTPUT),
           "--epochs", str(EPOCHS), "--lr", str(LEARNING_RATE)]
if SKIP_UNALIGNED_ACTIONS:
    command.append("--skip-unaligned-actions")
if RESUME and (OUTPUT/"last.json").exists():
    last = json.loads((OUTPUT/"last.json").read_text())
    command += ["--resume", str(OUTPUT/last["checkpoint"])]
print("Checkpoint output:", OUTPUT)
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
# 5. Xem best epoch, đường học và chất lượng alignment
import matplotlib.pyplot as plt
history = json.loads((OUTPUT/"history.json").read_text())
best = json.loads((OUTPUT/"best_weights.json").read_text())
print("Best epoch:", best["epoch"])
print("Validation MPJPE:", best["val_mpjpe_mm"], "mm")
print("Master baseline:", best["baseline_val_mpjpe_mm"], "mm")
print("Improved vs master:", best["val_mpjpe_mm"] < best["baseline_val_mpjpe_mm"])
print("Weights:", OUTPUT/"best_weights.json")
print("Inference config patch:", OUTPUT/"best_partwise.yml")
plt.plot([h["epoch"] for h in history], [h["train_mpjpe_mm"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val_mpjpe_mm"] for h in history], label="val")
plt.xlabel("Epoch"); plt.ylabel("MPJPE (mm)"); plt.legend(); plt.show()

import pandas as pd
alignment = pd.DataFrame(json.loads((OUTPUT/"alignment.json").read_text()))
display(alignment)
display(alignment[alignment["status"].isin(["rejected", "skipped_action"]) |
                  (alignment["coverage"] < 0.5) | (alignment["pa_mm"] > 80)])


## Các file kết quả
- epoch_0001.pt, epoch_0002.pt, ...: trọng số, optimizer, trạng thái RNG để tiếp tục train.
- epoch_0001_weights.json, ...: tham số từng epoch dùng trực tiếp cho partwise.
- best_weights.json: epoch có validation MPJPE thấp nhất, kèm epoch và baseline.
- best_partwise.yml: cấu hình inference khớp với train (visibility không có trong sheet).
- history.json, manifest.json, protocol.json, alignment.json: kết quả và nguồn dữ liệu để kiểm tra.

Đường dẫn trong best_partwise.yml trỏ đến Drive của Colab. Nếu tải kết quả về máy, sửa đường dẫn weight_model.path.
Ghép phần fusion trong file patch vào cấu hình pipeline hiện có; giữ các phần inputs/paths khác.
Notebook không train SMPL/WHAM backbone và không thay đổi sheet khi chạy.


In [ ]:
# 6. Ghép cấu hình tốt nhất vào pipeline để dùng inference
import copy, yaml
base = yaml.safe_load((REPO_DIR/"configs/pipeline.yml").read_text())
patch = yaml.safe_load((OUTPUT/"best_partwise.yml").read_text())
base["fusion"]["method"] = patch["fusion"]["method"]
base["fusion"]["partwise"] = patch["fusion"]["partwise"]
(OUTPUT/"pipeline_best.yml").write_text(yaml.safe_dump(base, sort_keys=False), encoding="utf-8")
print("Đã xuất:", OUTPUT/"pipeline_best.yml")
print("Điền inputs/paths đúng dữ liệu inference trước khi chạy pipeline.")
